In [1]:
from typing import cast, Sequence

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from pathlib import Path

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from core import Config
import pandas as pd

config = Config()
sector_numbers: Sequence[int] = range(10, 61, 5)

for i in sector_numbers:
    print(f"Training sector {i}")
    DATASET: str = f'basis_thresh_50_win_log_sector_{i}-r'
    DATA_PATH: Path = config.training_dir / f'{DATASET}.csv'
    # variables
    DEPTH: int = 4
    ITERATIONS: int = 500
    LEARNING_RATE: float = 0.1
    ESTIMATORS = 500
    FOLDS: int = 5
    SHUFFLE: bool = True
    STATE: int = 42
    #RESULTS_NAME: str = f'Linear_{DEPTH}_{DATASET}'
    RESULTS_NAME: str = f'GB_{DEPTH}_{DATASET}'
    if SHUFFLE:
        RESULTS_NAME += f'_sh{STATE}'

    all_data: pd.DataFrame = pd.read_csv(DATA_PATH, index_col=0)
    all_data.fillna(0, inplace=True)

    training_data = all_data.copy()
    training_data.drop(
        training_data[training_data["TR.UpstreamScope3PurchasedGoodsAndServices"].isna()].index,
        inplace=True
    )
    y: pd.Series = training_data['TR.UpstreamScope3PurchasedGoodsAndServices']
    X: pd.DataFrame = training_data.drop('TR.UpstreamScope3PurchasedGoodsAndServices', axis=1)
    # date could also be a categorical variable
    cat_cols: list[str] = ["Instrument", "TR.GICSSectorCode", "TR.HQCountryCode"]
    cat_cols.remove("Instrument")
    X[cat_cols] = X[cat_cols].astype("str")

    gkf: GroupKFold = GroupKFold(n_splits=FOLDS, random_state=STATE, shuffle=SHUFFLE)
    fold_results: list[dict] = []

    best_fold_index: int | None = None
    best_metric_value: float | None = None  # lower is better (RMSE)
    best_model: Pipeline = Pipeline(steps=[])
    best_X_val: pd.DataFrame | None = None
    best_y_val: pd.Series | None = None

    for fold_index, (train_index, val_index) in enumerate(gkf.split(X, y, groups=X['Instrument']), start=1):
        print(f"Fold {fold_index}")
        X_without_instrument = X.drop(columns=["Instrument"])
        X_train: pd.DataFrame = cast(pd.DataFrame, X_without_instrument.iloc[train_index])
        X_val: pd.DataFrame = cast(pd.DataFrame, X_without_instrument.iloc[val_index])
        y_train: pd.Series[float] = y.iloc[train_index]
        y_val: pd.Series[float] = y.iloc[val_index]

        numeric_cols = [c for c in X_train.columns if c not in cat_cols]

        preprocess = ColumnTransformer(
            transformers=[
                ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
                ("num", "passthrough", numeric_cols),
            ]
        )

        gb_reg = GradientBoostingRegressor(
            n_estimators=ESTIMATORS,
            learning_rate=LEARNING_RATE,
            max_depth=4,
            random_state=STATE,
        )

        lin_model = Pipeline(
            steps=[
                ("preprocess", preprocess),
                #("regressor", LinearRegression()),
                ("regressor", gb_reg),
            ]
        )

        lin_model.fit(X_train, y_train)

        y_val_pred_lin = lin_model.predict(X_val)
        y_val_pred_lin = np.asarray(y_val_pred_lin).ravel()

        rmse_lin = float(np.sqrt(mean_squared_error(y_val, y_val_pred_lin)))
        mae_lin = cast(float, mean_absolute_error(y_val, y_val_pred_lin))
        r2_lin = r2_score(y_val, y_val_pred_lin)

        fold_results.append(
            {
                #"model_name": "LinearRegression",
                "model_name": "GBRegression",
                "fold": fold_index,
                "sector": "All",
                "rmse": rmse_lin,
                "mae": mae_lin,
                "r2": r2_lin,
            }
        )

        # update best model based on RMSE
        if (best_metric_value is None) or (rmse_lin < best_metric_value):
            best_metric_value = rmse_lin
            best_fold_index = fold_index
            best_model = lin_model
            best_X_val = X_val.copy()
            best_y_val = y_val.copy()

        val_sectors: pd.Series = X_val["TR.GICSSectorCode"].astype("category")

        val_sectors = X_val["TR.GICSSectorCode"]
        val_df: pd.DataFrame = pd.DataFrame(
            {
                "y_true": y_val.to_numpy().ravel(),
                "y_pred": y_val_pred_lin,
                "sector": val_sectors.to_numpy().ravel(),
            }
        )

        for sector, grp in val_df.groupby("sector"):
            rmse = float(np.sqrt(mean_squared_error(grp["y_true"], grp["y_pred"])))
            mae = cast(float, mean_absolute_error(grp["y_true"], grp["y_pred"]))
            r2 = r2_score(grp["y_true"], grp["y_pred"])

            fold_results.append(
                {
                    "model_name": 'LinearRegression',
                    "fold": fold_index,
                    "sector": sector,
                    "rmse": rmse,
                    "mae": mae,
                    "r2": r2,
                }
            )

    print(f"Best fold by RMSE: {best_fold_index} (RMSE={best_metric_value:.2f})")
    metrics_df: pd.DataFrame = pd.DataFrame(fold_results)

    metrics_df.to_csv(config.results_dir / f'{RESULTS_NAME}_metrics.csv')

Training sector 10
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 2 (RMSE=1.19)
Training sector 15
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 5 (RMSE=1.12)
Training sector 20
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 4 (RMSE=1.65)
Training sector 25
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 5 (RMSE=1.34)
Training sector 30
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 2 (RMSE=1.78)
Training sector 35
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 1 (RMSE=1.31)
Training sector 40
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 4 (RMSE=2.14)
Training sector 45
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 1 (RMSE=1.52)
Training sector 50
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 5 (RMSE=1.99)
Training sector 55
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 2 (RMSE=2.21)
Training sector 60
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5
Best fold by RMSE: 1 (RMSE=1.94)
